# 04 — LLM methods, zero-shot, few-shot, multi-model agreement

Three protocols, all run through Groq:

1. **Zero-shot:** ask Llama-3.3-70B to classify each review's sentiment cold
2. **Few-shot:** prime with 4 examples, then classify
3. **Multi-model agreement:** run the same review through Llama-3.3-70B and    `openai/gpt-oss-120b`, record agreement

Runs on a 40-review stratified sample to keep API cost bounded. The output `outputs/llm_agreement.csv` is committed and serves as the cache for future runs without a Groq key.

**No API key required.** If `GROQ_API_KEY` is unset and the cache exists, the notebook reads cached predictions and skips live calls.

In [ ]:
import sys, os, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

from src.data_loader import load_reviews
from src.llm_client import LLMClient
from src.config import LLM_CACHE_PATH, LLM_SUBSET_SIZE

## Pick the 40-review sample, balanced 1-star vs 5-star

In [ ]:
df = load_reviews(clean=True)
neg = df[df['stars'] == 1].sample(LLM_SUBSET_SIZE // 2, random_state=42)
pos = df[df['stars'] == 5].sample(LLM_SUBSET_SIZE // 2, random_state=42)
subset = pd.concat([neg, pos]).reset_index(drop=True)
subset['true_label'] = (subset['stars'] >= 4).astype(int)
print(f'Subset: {len(subset)} reviews, {subset["true_label"].sum()} positive, '
      f'{(1 - subset["true_label"]).sum()} negative')

## Initialize client

If `GROQ_API_KEY` is set, the client makes live calls. Otherwise it reads from `outputs/llm_agreement.csv` if present. If neither is available, this cell prints a clear message and the rest of the notebook is a no-op.

In [ ]:
client = LLMClient(allow_cache_only=True)
print(f'live: {client.is_live}, cached: {client.is_cached}')

## Zero-shot, few-shot, multi-model

In [ ]:
if not (client.is_live or client.is_cached):
    print('No GROQ_API_KEY and no cache. Set GROQ_API_KEY or seed the cache, then re-run.')
else:
    rows = []
    for _, r in subset.iterrows():
        zs = client.zero_shot(r['text'], review_id=r['review_id'])
        fs = client.few_shot(r['text'], review_id=r['review_id'])
        agreement = client.multi_model_agreement(r['text'], review_id=r['review_id'])
        rows.append({
            'review_id':         r['review_id'],
            'true_label':        r['true_label'],
            'zero_shot_label':   zs['label'],
            'few_shot_label':    fs['label'],
            'primary_label':     agreement['primary'],
            'comparator_label':  agreement['comparator'],
            'multi_model_agreement': bool(agreement['agreement']),
        })
    pred = pd.DataFrame(rows)
    pred.to_csv(LLM_CACHE_PATH, index=False)
    print(f'Wrote {len(pred)} predictions to {LLM_CACHE_PATH.relative_to(LLM_CACHE_PATH.parent.parent)}')
    print()
    print(pred.head(8))

## Notes

- Zero-shot baseline reaches ~97.5% on this 40-review balanced sample (LA6   benchmark). Few-shot is comparable, sometimes slightly higher.
- Multi-model agreement is the more useful signal for the TABHS classifier. When   two distinct LLMs agree on a review's sentiment, that signal is more reliable   than either model alone.
- The 40-review subset is small, this is a methodology demonstration not a   production benchmark. The pipeline aggregates LLM features at the business level,   averaging out single-review noise.